# Part 1: Data Preparation

Keep the raw OHLCV fields so price ranges, trading volume and adjustment factors can be audited.
Model returns use Adj Close, which includes the provider's split/distribution adjustments;
ordinary Close can mistake a cash distribution for an investment loss.
SPY, QQQ and IWM cover US equities; TLT and HYG cover Treasury and high-yield bonds;
GLD covers gold and DBC covers commodity futures.

The requested start (2010-01-01) and inclusive cutoff (2026-09-16) are project choices.
Existing raw snapshots are hash-checked and never overwritten. A fixed cutoff alone cannot
reproduce a particular historical vintage if Yahoo later revises prices.
This script is the authoritative implementation; the notebook is generated by src/sync_notebooks.py.

In [1]:
from pathlib import Path
from IPython.display import display
import hashlib
import json
import numpy as np
import pandas as pd
import sys


def find_project_root():
    for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (candidate / 'notebooks' / '01_data_preparation.ipynb').is_file() and (candidate / 'src').is_dir():
            return candidate
    raise RuntimeError('Start Jupyter inside the project root or one of its subdirectories.')

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from data_preparation_checks import check_trading_dates, check_raw_bars

PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
REPORT_DIR = PROJECT_ROOT / 'outputs' / 'tables'
for directory in (PROCESSED_DIR, RAW_DIR, REPORT_DIR):
    assert directory.resolve().is_relative_to(PROJECT_ROOT)
    directory.mkdir(parents=True, exist_ok=True)
print('Project root verified.')

import platform
from datetime import datetime, timezone
import yfinance as yf

TICKERS = ['SPY', 'QQQ', 'IWM', 'TLT', 'HYG', 'GLD', 'DBC']
START_DATE = '2010-01-01'
DATA_CUTOFF = '2026-09-16'
END_DATE_EXCLUSIVE = '2026-09-17'
SNAPSHOT_PATH = RAW_DIR / 'yahoo_ohlcv_20100101_20260916.csv'
METADATA_PATH = RAW_DIR / 'yahoo_ohlcv_20100101_20260916_metadata.json'
# Keep the provider cache inside the project as well.
yf.set_tz_cache_location(str(RAW_DIR / 'yfinance_cache'))

Project root verified.


## 1. Download or Load the Immutable Raw Snapshot

In [2]:
snapshot_exists = SNAPSHOT_PATH.exists()
if snapshot_exists:
    metadata = json.loads(METADATA_PATH.read_text())
    assert hashlib.sha256(SNAPSHOT_PATH.read_bytes()).hexdigest() == metadata['sha256']
    assert metadata['tickers'] == TICKERS and metadata['end_exclusive'] == END_DATE_EXCLUSIVE
    assert metadata['start'] == START_DATE and metadata['cutoff_inclusive'] == DATA_CUTOFF
    raw_data = pd.read_csv(SNAPSHOT_PATH, header=[0, 1], index_col=0, parse_dates=True, float_precision='round_trip')
    print('Loaded verified raw snapshot:', SNAPSHOT_PATH.relative_to(PROJECT_ROOT))
else:
    raw_data = yf.download(TICKERS, start=START_DATE, end=END_DATE_EXCLUSIVE,
                           auto_adjust=False, keepna=True, progress=False, threads=False)
    assert not raw_data.empty, 'Download failed; do not substitute old processed data.'
    assert 'Adj Close' in raw_data.columns.get_level_values(0)
    assert set(TICKERS).issubset(raw_data['Adj Close'].columns)
    assert raw_data['Adj Close'][TICKERS].notna().any().all(), 'An asset download failed.'

# Validate either source once, before a new snapshot can be saved.
calendar_audit = check_trading_dates(raw_data.index, START_DATE, DATA_CUTOFF)
raw_bar_audit = check_raw_bars(raw_data, TICKERS)
if not snapshot_exists:
    raw_data.to_csv(SNAPSHOT_PATH, index_label='Date')
    metadata = dict(source='Yahoo Finance via yfinance', downloaded_at_utc=datetime.now(timezone.utc).isoformat(),
                    start=START_DATE, cutoff_inclusive=DATA_CUTOFF, end_exclusive=END_DATE_EXCLUSIVE,
                    tickers=TICKERS, auto_adjust=False, keepna=True,
                    versions=dict(python=platform.python_version(), pandas=pd.__version__, numpy=np.__version__, yfinance=yf.__version__),
                    sha256=hashlib.sha256(SNAPSHOT_PATH.read_bytes()).hexdigest())
    METADATA_PATH.write_text(json.dumps(metadata, indent=2))

print('Trading-calendar check:', calendar_audit)
print('Raw OHLCV and Adj Close checks passed for all seven ETFs.')
prices = raw_data['Adj Close'].reindex(columns=TICKERS).copy()
prices.index = pd.DatetimeIndex(prices.index).tz_localize(None)
prices.index.name = 'Date'
assert prices.index.notna().all()
assert prices.index.min() >= pd.Timestamp(START_DATE)
assert prices.index.max() <= pd.Timestamp(DATA_CUTOFF)
print('Raw adjusted-price shape:', prices.shape)

Loaded verified raw snapshot: data\raw\yahoo_ohlcv_20100101_20260916.csv


Trading-calendar check: {'calendar': 'XNYS', 'package_version': '4.13.2', 'expected_sessions': 4201, 'expected_first': '2010-01-04', 'expected_last': '2026-09-16', 'weekend_dates': 1744, 'regular_holiday_dates': 154, 'special_closures': ['2012-10-29', '2012-10-30', '2018-12-05', '2025-01-09'], 'missing_sessions': 0, 'unexpected_dates': 0}
Raw OHLCV and Adj Close checks passed for all seven ETFs.
Raw adjusted-price shape: (4201, 7)


## 2. Retain Complete, Validated Prices

XNYS from exchange_calendars is the US cash-equity session reference for these US-listed ETFs,
including the bond and commodity ETFs (not their underlying bond/futures calendars).
Weekends, regular holidays and special closures are separated; early-close sessions remain required.
Raw dates must cover the full requested session range; the price table retains these dates.
Sources: [calendar rules](https://github.com/gerrymanoim/exchange_calendars/blob/master/exchange_calendars/exchange_calendar_xnys.py)
and [NYSE calendar](https://www.nyse.com/trade/hours-calendars).

Missing/non-finite raw fields or invalid prices stop execution for investigation. No price is
filled, fabricated or dropped. Complete Adj Close prices pass through unchanged.

In [3]:
def audit_frame(frame, *, is_returns=False):
    values = frame.to_numpy(dtype=float)
    audit = dict(rows=len(frame), columns=frame.shape[1], start_date=str(frame.index.min().date()),
                end_date=str(frame.index.max().date()), missing_values=int(frame.isna().sum().sum()),
                duplicate_dates=int(frame.index.duplicated().sum()), infinite_values=int(np.isinf(values).sum()),
                chronological=bool(frame.index.is_monotonic_increasing))
    if is_returns:
        # Negative and zero returns are valid observations, not data errors.
        audit.update(zero_returns=int((values == 0).sum()), negative_returns=int((values < 0).sum()))
    else:
        audit['non_positive_values'] = int((values <= 0).sum())
    return audit

before = audit_frame(prices)
asset_audit = pd.DataFrame({
    'start_date': prices.apply(lambda s: s.first_valid_index()),
    'end_date': prices.apply(lambda s: s.last_valid_index()),
    'observed_prices': prices.count(), 'missing_before': prices.isna().sum()
})
# Keep the existing report format; a successful raw check guarantees no missing observations.
missing_audit = pd.DataFrame(columns=['Date', 'asset', 'reason'])
missing_audit.to_csv(REPORT_DIR / 'part1_missing_observations.csv', index=False)
clean_prices = prices.copy()
asset_audit['missing_after'] = clean_prices.isna().sum()
asset_audit.to_csv(REPORT_DIR / 'part1_asset_audit.csv', index_label='asset')
display(asset_audit)
display(missing_audit)
print('Before cleaning:', before)
print('After cleaning:', audit_frame(clean_prices))
print('All validated prices retained; no missing values filled or dates removed.')

,start_date,end_date,observed_prices,missing_before,missing_after
Ticker,,,,,
SPY,2010-01-04,2026-09-16,4201,0,0
QQQ,2010-01-04,2026-09-16,4201,0,0
IWM,2010-01-04,2026-09-16,4201,0,0
TLT,2010-01-04,2026-09-16,4201,0,0
HYG,2010-01-04,2026-09-16,4201,0,0
GLD,2010-01-04,2026-09-16,4201,0,0
DBC,2010-01-04,2026-09-16,4201,0,0


,Date,asset,reason


Before cleaning: {'rows': 4201, 'columns': 7, 'start_date': '2010-01-04', 'end_date': '2026-09-16', 'missing_values': 0, 'duplicate_dates': 0, 'infinite_values': 0, 'chronological': True, 'non_positive_values': 0}
After cleaning: {'rows': 4201, 'columns': 7, 'start_date': '2010-01-04', 'end_date': '2026-09-16', 'missing_values': 0, 'duplicate_dates': 0, 'infinite_values': 0, 'chronological': True, 'non_positive_values': 0}
All validated prices retained; no missing values filled or dates removed.


## Adjustment diagnostics

A large level gap between Close and Adj Close often accumulates over years of distributions;
it is not itself a bad price. We also flag daily changes of at least 1% in Adj Close / Close.
This is a review threshold, not a deletion rule. Yahoo's Close may already reflect splits,
so this ratio alone cannot identify every corporate action.

In this snapshot the flagged events are DBC adjustments. Cash distributions are a plausible
explanation; the original snapshot has no corporate-action fields to verify every event.
The largest event, 2024-12-23, is consistent with DBC's reported $1.11582 distribution:
[Invesco 2024 annual report](https://www.invesco.com/us-rest/contentdetail?contentId=65d1a35f-2a2d-40ec-a7c2-c34239473cec&dnsName=us).
Dates and adjusted prices are retained without replacement. This audit does not establish
that every vendor adjustment is correct or available as it was known historically.

In [4]:
adjustment_factor = prices / raw_data['Close'][TICKERS]
factor_change = adjustment_factor.pct_change(fill_method=None)
adjustment_summary = pd.DataFrame({
    'min_adj_close_to_close': adjustment_factor.min(),
    'max_adj_close_to_close': adjustment_factor.max(),
    'largest_absolute_factor_change': factor_change.abs().max()
})
adjustment_events = []
for asset in TICKERS:
    for date in factor_change.index[factor_change[asset].abs().ge(0.01)]:
        adjustment_events.append(dict(Date=str(date.date()), asset=asset,
                                     factor_change=float(factor_change.loc[date, asset]),
                                     assessment='Adjustment-factor change; possible distribution or provider revision. Retained.'))
display(adjustment_summary)
display(pd.DataFrame(adjustment_events, columns=['Date', 'asset', 'factor_change', 'assessment']))

,min_adj_close_to_close,max_adj_close_to_close,largest_absolute_factor_change
Ticker,,,
SPY,0.744454,0.997523,0.007093
QQQ,0.867008,1.000000,0.005570
IWM,0.798862,1.000000,0.008661
TLT,0.613192,1.000000,0.004279
HYG,0.382786,1.000000,0.007748
GLD,1.000000,1.000000,0.000000
DBC,0.845437,1.000000,0.053387


,Date,asset,factor_change,assessment
0,2018-12-24,DBC,0.012989,Adjustment-factor change; possible distributio...
1,2019-12-23,DBC,0.016019,Adjustment-factor change; possible distributio...
2,2023-12-18,DBC,0.049140,Adjustment-factor change; possible distributio...
3,2024-12-23,DBC,0.053387,Adjustment-factor change; possible distributio...
4,2025-12-22,DBC,0.033656,Adjustment-factor change; possible distributio...


## 3. Calculate and Validate Daily Returns

For each asset, `return[t] = price[t] / price[t-1] - 1`. Only the first row lacks a prior observed price. A genuine unchanged price may produce a zero return; no zeros are manufactured by filling missing prices.

In [5]:
untrimmed_returns = clean_prices.pct_change(fill_method=None)
assert untrimmed_returns.iloc[0].isna().all()
daily_returns = untrimmed_returns.iloc[1:].copy()
assert daily_returns.index.equals(clean_prices.index[1:])
for frame in (clean_prices, daily_returns):
    assert frame.index.is_unique and frame.index.is_monotonic_increasing
    assert np.isfinite(frame.to_numpy()).all()
np.testing.assert_allclose(daily_returns, clean_prices.to_numpy()[1:] / clean_prices.to_numpy()[:-1] - 1)
print('Initial return NaNs (expected):', int(untrimmed_returns.isna().sum().sum()))
print('Validated daily returns:', audit_frame(daily_returns, is_returns=True))
print('Observed zero returns by asset:')
display(daily_returns.eq(0).sum().to_frame('zero_returns'))
print('Daily return shape:', daily_returns.shape)

Initial return NaNs (expected): 7
Validated daily returns: {'rows': 4200, 'columns': 7, 'start_date': '2010-01-05', 'end_date': '2026-09-16', 'missing_values': 0, 'duplicate_dates': 0, 'infinite_values': 0, 'chronological': True, 'zero_returns': 206, 'negative_returns': 13475}
Observed zero returns by asset:


,zero_returns
Ticker,
SPY,12
QQQ,16
IWM,6
TLT,15
HYG,64
GLD,13
DBC,80


Daily return shape: (4200, 7)


## 4. Save Processed Data and the Audit Trail

In [6]:
clean_prices.to_csv(PROCESSED_DIR / 'clean_adjusted_close_prices.csv', index_label='Date')
daily_returns.to_csv(PROCESSED_DIR / 'daily_returns.csv', index_label='Date')
report = dict(raw_adjusted_prices=before, cleaned_prices=audit_frame(clean_prices),
              calendar_audit=calendar_audit, raw_bar_audit=raw_bar_audit,
              adjustment_factor_summary=adjustment_summary.to_dict(orient='index'),
              adjustment_review_threshold=0.01, adjustment_events=adjustment_events,
              daily_returns=audit_frame(daily_returns, is_returns=True), removed_price_rows=len(prices)-len(clean_prices),
              expected_initial_return_nans=len(TICKERS), raw_snapshot_sha256=metadata['sha256'],
              cutoff=DATA_CUTOFF, processed_sha256={name: hashlib.sha256((PROCESSED_DIR / name).read_bytes()).hexdigest()
                  for name in ['clean_adjusted_close_prices.csv', 'daily_returns.csv']})
(REPORT_DIR / 'part1_quality_report.json').write_text(json.dumps(report, indent=2))
print('Part 1 completed successfully. Raw snapshot and audit reports retained.')

Part 1 completed successfully. Raw snapshot and audit reports retained.
